# Diffuse scattering with PyTorch

Run **Kernel → Restart Kernel and Run All Cells** for a CPU simulation of the bundled histidine crystal, inline slices, and phonon diagnostics. No precomputed data is required. Change the inputs below to explore another structure.

Use the Python environment containing the repository requirements, JupyterLab, and ipykernel. See the [quickstart guide](../docs/PYTORCH_QUICKSTART.md) for installation and the equivalent [command-line script](pytorch_diffuse.py). The existing [pump–probe notebook](../pump_probe_validation.ipynb) is a separate, larger experiment.

Outputs go to `tmp/pytorch-notebook/`; re-running overwrites that directory's example files. This does not change the command-line example's outputs.

In [ ]:
from pathlib import Path
import sys

# Jupyter may start the kernel in the repository root or in examples/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "eryx/models_torch.py").is_file()
             and (p / "tests/pdbs/histidine.pdb").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the eryx checkout")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%matplotlib inline
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic
from eryx.models_torch import OnePhonon
from eryx.visuals import visualize_central_slices

## Choose inputs

`SAMPLING` is (minimum, maximum, **oversampling**), so `(-2, 2, 3)` gives 13 points per axis, not three. Start on CPU with histidine. For a protein, change `PDB` to `ROOT / "tests/pdbs/6o2h_clean.pdb"`; expect greater memory and runtime. A custom PDB needs crystal cell and symmetry information.

The model cell also exposes the network cutoff and spring strengths. After changing inputs, rerun the input cell and all following cells, or use Run All.

In [ ]:
PDB = ROOT / "tests/pdbs/histidine.pdb"
SAMPLING = (-2, 2, 3)
DEVICE = torch.device("cpu")
OUT = ROOT / "tmp/pytorch-notebook"
assert PDB.is_file(), f"Missing PDB: {PDB}"

## Construct the model

The constructor builds the crystal, elastic network, displacement modes, and covariance. `group_by="asu"` projects onto six rigid-body degrees of freedom per asymmetric unit. `res_limit=0.0` leaves the high-resolution cutoff disabled. The import above explicitly selects PyTorch; `from eryx import OnePhonon` would select NumPy.

In [ ]:
model = OnePhonon(
    str(PDB), hsampling=SAMPLING, ksampling=SAMPLING, lsampling=SAMPLING,
    model="gnm", group_by="asu", expand_p1=True, res_limit=0.0,
    gnm_cutoff=4.0, gamma_intra=1.0, gamma_inter=1.0,
    device=DEVICE, n_processes=1,
)

## Calculate diffuse intensity

`use_data_adp=True` uses the PDB B-factors for displacement damping. Set it to False to use computed ADPs, which are rescaled against the PDB values. Keep the returned `intensity` tensor for PyTorch calculations; the detached NumPy arrays below are for plotting and export.

Preserve NaNs from singular modes or resolution masking. Use a common finite mask when comparing runs, and reshape only with the model's recorded grid shape.

In [ ]:
intensity = model.apply_disorder(use_data_adp=True)
I = intensity.detach().cpu().numpy()
q = model.q_grid.detach().cpu().numpy()
hkl = model.hkl_grid.detach().cpu().numpy()
volume = I.reshape(model.map_shape)
valid = np.isfinite(I)
assert I.size == np.prod(model.map_shape)
assert valid.any() and np.max(I[valid]) > 0

print("Grid:", model.map_shape, "finite:", valid.sum(), "/", I.size)
print("Intensity min/mean/max:", I[valid].min(), I[valid].mean(), I[valid].max())

## Save the result

The NPZ records intensities, Cartesian q vectors (Å⁻¹, including 2π), Miller coordinates, and grid shape in matching point order. It also saves **model-derived** ADP and covariance diagnostics; those ADPs are not the PDB values used for damping above. With rigid-body grouping, covariance is in rigid-body coordinates, not an atom-by-atom matrix.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    OUT / "results.npz", intensity=I, q_vectors=q, hkl_grid=hkl,
    map_shape=model.map_shape,
    adp=model.ADP.detach().cpu().numpy(),
    covariance=model.covar.detach().cpu().numpy(),
)
print("Saved:", OUT.relative_to(ROOT) / "results.npz")

## Central slices

These are the h=0, k=0, and l=0 planes for the symmetric sampling above. White regions are missing data. The existing helper hides tick labels; these are array views, not calibrated Cartesian q plots.

In [ ]:
visualize_central_slices(volume, output=str(OUT / "slices.png"))
plt.show()

## Radial intensity and mode density

The radial profile averages finite voxels in |q| bins. Outer bins may contain only part of a spherical shell because the sampled region is a box.

The frequency helper uses an **empirical scale**, not an experimentally calibrated THz conversion. The histogram is mode density, not a thermal population spectrum. See [PDOS interpretation](../docs/PDOS.md). Eigenvectors are detached in the phonon code, so autograd does not provide every structural derivative.

In [ ]:
qnorm = np.linalg.norm(q, axis=1)
profile, edges, _ = binned_statistic(qnorm[valid], I[valid],
                                    statistic="mean", bins=20)
pdos = model.generate_pdos(bins=30)
np.savetxt(OUT / "pdos.dat", pdos, header="frequency_THz_empirical density")
np.savetxt(OUT / "radial.dat", np.column_stack(((edges[1:]+edges[:-1])/2, profile)),
           header="q_inverse_angstrom mean_intensity")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot((edges[1:]+edges[:-1])/2, profile)
axes[0].set(xlabel="|q| (1/Angstrom)", ylabel="Mean diffuse intensity")
axes[1].plot(pdos[:, 0], pdos[:, 1])
axes[1].set(xlabel="Frequency (THz; empirical scale)", ylabel="Mode density")
fig.tight_layout()
fig.savefig(OUT / "diagnostics.png", dpi=150)
plt.show()

## Check the saved outputs

The model, tensor, and NumPy arrays remain available for inspection. For another run, change the inputs and rerun the input cell and all following cells. Interactive volumes and population-change examples are linked from the [guide](../docs/PYTORCH_QUICKSTART.md).

In [ ]:
with np.load(OUT / "results.npz") as saved:
    np.testing.assert_allclose(saved["intensity"], I, equal_nan=True)
    assert tuple(saved["map_shape"]) == model.map_shape
    assert saved["q_vectors"].shape == (I.size, 3)
assert np.isfinite(pdos).all()
for name in ("results.npz", "slices.png", "diagnostics.png", "radial.dat", "pdos.dat"):
    path = OUT / name
    assert path.is_file() and path.stat().st_size > 0, name
    print(path.relative_to(ROOT))